In [2]:
%load_ext autoreload
%autoreload 2

import itertools
import math
from math import pi

import matplotlib.pyplot as plt
import numpy as np
import scipy.signal
import tqdm.contrib.itertools
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.spectrum.autodiff import FunctionComposition, SampledFunction, FunctionSum
from fluxoniumcr.spectrum.planck import PlanckRampFunction
from fluxoniumcr.spectrum.square_spectrum import calculate_square_spectrum
from fluxoniumcr.utils import load_arguments

In [3]:
charge_op_parent_path = DATA_DIR/"charge_operator/EJ=4.00,EC=1.20,EL=0.40"
charge_op_path = charge_op_parent_path/"driven_charge_operator.hdf5"
amp_from_deltap_path = charge_op_parent_path/"amplitude_from_deltap.hdf5"

charge_op_dataset = xr.load_dataset(charge_op_path)
amp_from_deltap_dataset = xr.load_dataset(amp_from_deltap_path)

Ω0 = charge_op_dataset.amplitude.attrs['plot_unit']

In [4]:
from fluxoniumcr.spectrum.autodiff import SampledFunction


def create_sampled_function(x, y, window_length=5) -> SampledFunction:
    dx = x[1] - x[0]
    
    ys = [y]
    
    for i in range(1, 3):
        yi = scipy.signal.savgol_filter(
            y,
            window_length,
            polyorder=2,
            delta=dx,
            deriv=i,
            # We assume the function is symmetric around x=0.
            mode='mirror',
        )
        # Linearly extrapolate numerical derivative at domain end.
        k = window_length//2 + 1
        dyi = yi[-k] - yi[-k-1]
        yi[-k:] = yi[-k] + np.arange(k) * dyi
        ys.append(yi)

    func = SampledFunction(x, *ys)
    
    return func

In [5]:
charge_op = xr.DataArray(
    None,
    coords=[
        charge_op_dataset.frequency,
        charge_op_dataset.harmonic[abs(charge_op_dataset.harmonic) <= 8],
        xr.DataArray([0, 1], dims=['ket']),
        charge_op_dataset.bra,
    ],
)

quasienergy = xr.DataArray(
    None,
    coords=[
        charge_op_dataset.frequency,
        charge_op_dataset.ket,
    ],
)

In [30]:
for frequency, harmonic, bra, ket in tqdm.contrib.itertools.product(
        charge_op.frequency.data,
        charge_op.harmonic.data,
        charge_op.bra.data,
        charge_op.ket.data,
):
    key = dict(
        frequency=frequency,
        harmonic=harmonic,
        bra=bra,
        ket=ket,
    )
    ds = charge_op_dataset.loc[key]
    amplitude_data = ds.amplitude.data
    matrix_element_data = ds.matrix.data
    
    if ds.attrs['phase_gauge'] and bra == ket and abs(harmonic) == 1:
        # Convert to charge driving.
        matrix_element_data = (
            matrix_element_data
            + np.sign(harmonic)
                * amplitude_data/(16*ds.attrs['EC'])
        )
    
    matrix_element_func = create_sampled_function(amplitude_data, matrix_element_data)
    charge_op.loc[key] = matrix_element_func
    
    
for frequency, ket in tqdm.contrib.itertools.product(
        quasienergy.frequency.data,
        quasienergy.ket.data,
):
    key = dict(
        frequency=frequency,
        ket=ket,
    )
    ds = charge_op_dataset.loc[key]
    amplitude_data = ds.amplitude.data
    quasienergy_data = ds.quasienergy.data
    quasienergy_func = create_sampled_function(amplitude_data, quasienergy_data)
    quasienergy.loc[key] = quasienergy_func

  0%|          | 0/72096 [00:00<?, ?it/s]

  0%|          | 0/4506 [00:00<?, ?it/s]

In [43]:
ramp_duration = 50.0
ramp_shape = "planck"
ramp_dt = 0.1

fourier_frequency_data = np.float32(16 * (np.arange(2**11) - 2**10)/(2**10))
harmonic_data = charge_op.harmonic.data

In [84]:
# Calculate ESD for one deltap and sweeping drive frequency
deltap_data = [0.6, 0.8, 1.0]

drive_frequency_data = charge_op.frequency.data
amplitude_data = amp_from_deltap_dataset.amplitude_spline.sel(deltap=deltap_data).data

# ====================

# Calculate ESD for one drive frequency and sweeping drive amplitude
drive_frequency_data = np.array([0.502 * 2*pi])
amplitude_data = np.linspace(0.0, 0.8, 81)[None] * Ω0

p_data = [
    charge_op_dataset.matrix
        .sel(harmonic=-1, bra=i, ket=i)
        .interp(
            frequency=drive_frequency_data.squeeze(),
            amplitude=amplitude_data.squeeze(),
        )
    for i in [0, 1]
]
n01 = charge_op_dataset.matrix.isel(frequency=0, amplitude=0).sel(harmonic=-1, bra=0, ket=1).item()
deltap_data = (abs(p_data[1] - p_data[0])/n01).data.squeeze()

In [85]:
dataset = xr.Dataset(
    attrs=dict(
        ramp_shape=ramp_shape,
        ramp_duration=ramp_duration,
        ramp_dt=ramp_dt,
    )
)

drive_frequency_coord = xr.DataArray(
    drive_frequency_data,
    dims=['drive_frequency'],
    attrs=dict(
        long_name="Drive frequency",
        units="Grad/s",
    )
)
deltap_coord = xr.DataArray(
    deltap_data,
    dims=['deltap'],
)

dataset['amplitude'] = xr.DataArray(
    amplitude_data,
    coords=[drive_frequency_coord, deltap_coord],
    attrs=dict(
        long_name="Midpoint pulse amplitude",
        plot_unit=Ω0,
        units="Grad/s",
    )
)

harmonic_coord = xr.DataArray(
    harmonic_data,
    dims=['harmonic'],
    attrs=dict(
        long_name="Transition photon number",
    )
)

frequency_coord = charge_op_dataset.frequency.copy(deep=True)

ket_coord = charge_op_dataset.ket.sel(ket=[0, 1])
bra_coord = charge_op_dataset.bra.copy(deep=True)

fourier_frequency_coord = xr.DataArray(
    fourier_frequency_data,
    dims=['fourier_frequency'],
    attrs=dict(
        long_name="Fourier frequency",
        units="Grad/s",
    )
)

dataset['numerator'] = xr.DataArray(
    np.float32('nan'),
    coords=[
        drive_frequency_coord,
        deltap_coord,
        bra_coord,
        ket_coord,
        harmonic_coord,
        fourier_frequency_coord,
    ],
    attrs=dict(
        long_name="Spectrum numerator",
        comment="Centered around the bare transition frequency.",
    )
)
dataset['pole'] = xr.DataArray(
    np.float32('nan'),
    coords=[
        drive_frequency_coord,
        deltap_coord,
        bra_coord,
        ket_coord,
        harmonic_coord,
    ],
    attrs=dict(
        long_name="Dressed transition frequency",
        units="Grad/s",
    )
)
dataset['bare_pole'] = xr.DataArray(
    np.float32('nan'),
    coords=[
        drive_frequency_coord,
        bra_coord,
        ket_coord,
        harmonic_coord,
    ],
    attrs=dict(
        long_name="Bare transition frequency",
        units="Grad/s",
    )
)

In [87]:
for (
        drive_frequency,
        deltap,
        harmonic,
        bra,
        ket,
) in tqdm.contrib.itertools.product(
        dataset.drive_frequency.data,
        dataset.deltap.data,
        dataset.harmonic.data,
        dataset.bra.data,
        dataset.ket.data,
):
    ds = dataset.loc[dict(
        drive_frequency=drive_frequency,
        deltap=deltap,
        bra=bra,
        ket=ket,
        harmonic=harmonic,
    )]
    
    drive_amplitude = ds.amplitude.item()
    
    if np.isnan(drive_amplitude): continue
        
    # If data already exists, then skip re-calculating it.
    if not ds.pole.isnull().any(): continue
        
    matrix_element_func = charge_op.sel(
        frequency=drive_frequency,
        harmonic=harmonic,
        bra=bra,
        ket=ket,
    ).item()
    
    bra_energy_func = quasienergy.sel(frequency=drive_frequency, ket=bra).item()
    ket_energy_func = quasienergy.sel(frequency=drive_frequency, ket=ket).item()
    delta_energy_func = FunctionSum(
        [bra_energy_func, ket_energy_func],
        coeffs=(1, -1),
        constant=harmonic * drive_frequency,
    )

    ts = ds.ramp_dt * np.arange(1 + math.ceil(ds.ramp_duration/ds.ramp_dt))
    
    if ds.ramp_shape == "planck":
        ramp_func = PlanckRampFunction(ds.ramp_duration, drive_amplitude)
    else:
        raise ValueError(ds.ramp_shape)

    envelope_func = FunctionComposition(matrix_element_func, ramp_func)
    instfreq_func = FunctionComposition(delta_energy_func, ramp_func)

    spectrum = calculate_square_spectrum(
        envelope_func(ts),
        envelope_func.derivative(1)(ts),
        envelope_func.derivative(2)(ts),
        instfreq_func(ts),
        instfreq_func.derivative(1)(ts),
        ts[1] - ts[0],
    )
    
    ds.numerator[:] = spectrum.numerator(spectrum.pole + ds.fourier_frequency)
    ds.pole[()] = spectrum.pole
    ds.bare_pole[()] = spectrum.bare_pole

  0%|          | 0/216288 [00:00<?, ?it/s]

In [96]:
dataset.to_netcdf(charge_op_parent_path/"esd.hdf5")